# Task 3: Forecasting Models — Training

This notebook trains three models on each of the three target areas:
- **SARIMA** — Classical statistical model with seasonal support
- **LSTM** — Recurrent neural network with gated memory
- **Transformer** — Self-attention-based sequence model

**Test period:** December 16–22 (strictly held out — never used for training or validation)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import pickle

from src.data.loader import (
    load_area_from_parquet, get_area_series, total_traffic_per_area,
)
from src.data.preprocessor import (
    train_test_split_by_date, fit_scaler, scale_series,
    inverse_scale, create_sequences, fill_missing,
)
from src.models.sarima_model import run_sarima
from src.models.lstm_model import run_lstm
from src.models.transformer_model import run_transformer
from src.utils.timer import get_hardware_info

PARQUET_DIR = '../data/processed/milan_traffic_parquet/'
TOTALS_PATH = '../data/processed/total_per_area.parquet'
MODELS_DIR  = '../outputs/models/'
os.makedirs(MODELS_DIR, exist_ok=True)

SEQ_LENGTH = 144  # 1 day = 144 x 10-min intervals

print('Hardware info:')
for k, v in get_hardware_info().items():
    print(f'  {k}: {v}')

## 3.1 Identify Target Areas

In [ ]:
if os.path.exists(TOTALS_PATH):
    totals = pd.read_parquet(TOTALS_PATH).iloc[:, 0]
    totals.index = totals.index.astype('int32')
else:
    totals = total_traffic_per_area(PARQUET_DIR)
    totals.to_frame().to_parquet(TOTALS_PATH)

top_area = int(totals.idxmax())
TARGET_AREAS = {
    f'Area_{top_area}_TopTraffic': top_area,
    'Area_4159': 4159,
    'Area_4556': 4556,
}
print('Target areas:', TARGET_AREAS)

## 3.2 Preprocessing Pipeline

For each area:
1. Extract time series
2. Fill missing timesteps via linear interpolation
3. Split: train = all data before Dec 16, test = Dec 16–22
4. Fit MinMaxScaler on train only, apply to test
5. Build overlapping sequences of length 144 for neural models

In [ ]:
# Selectively read the 3 target areas (predicate pushdown — no full-dataset load)
df_3 = load_area_from_parquet(PARQUET_DIR, list(TARGET_AREAS.values()))
df_3['datetime'] = pd.to_datetime(df_3['time_interval'], unit='ms')
print(f'Loaded {len(df_3):,} rows for {df_3["square_id"].nunique()} areas')

area_data = {}
for label, sq_id in TARGET_AREAS.items():
    series = get_area_series(df_3, sq_id)
    series = fill_missing(series)

    train, test = train_test_split_by_date(series)
    train_scaled, scaler = fit_scaler(train)
    test_scaled = scale_series(test, scaler)

    # Sequences for neural models — built over the concatenated stream so the
    # first test sample's context is the tail of the train window.
    full_scaled = np.concatenate([train_scaled, test_scaled])
    X_all, y_all = create_sequences(full_scaled, SEQ_LENGTH)

    n_train = len(train_scaled) - SEQ_LENGTH
    X_train, y_train = X_all[:n_train], y_all[:n_train]
    X_test,  y_test  = X_all[n_train:], y_all[n_train:]
    y_test_orig = inverse_scale(y_test, scaler)

    area_data[label] = {
        'train': train, 'test': test, 'scaler': scaler,
        'X_train': X_train, 'y_train': y_train,
        'X_test':  X_test,  'y_test_orig': y_test_orig,
    }
    print(f'{label}: train={len(train)}, test={len(test)}, X_train={X_train.shape}')

## 3.2b Baselines: persistence & seasonal-naive

Anchor the comparison with two zero-learning baselines so the trained
models can be judged against a non-trivial floor. Persistence
($\\hat{y}_{t+1}=y_t$) is what a SARIMA without seasonality would degrade
to; seasonal-naive ($\\hat{y}_{t+1}=y_{t+1-144}$) embeds the strong daily
cycle that ACF Lag-144 in Task 2 already exposed.

In [ ]:
from src.models.baseline import run_persistence, run_seasonal_naive

persistence_results    = {}
seasonal_naive_results = {}
for label, data in area_data.items():
    print(f'\n--- {label} ---')
    persistence_results[label]    = run_persistence(data['train'], data['test'])
    seasonal_naive_results[label] = run_seasonal_naive(
        data['train'], data['test'], period=144
    )

## 3.2c Hyperparameter selection

**SARIMA.** AIC grid over a small (p,d,q)(P,D,Q,144) cube, fit on the last
7 days of the training set (subsampled to keep search time manageable —
each fit at s=144 already costs tens of seconds). Best order is re-fit on
the full training series in section 3.3.

**LSTM / Transformer.** Small random search on a validation tail of the
training series for *one* representative area (the top-traffic cell); the
chosen config is then reused across all three areas to keep total compute
tractable. This is documented as a deliberate trade-off.

In [ ]:
from src.models.tuning import sarima_aic_search

# Tune on the top-traffic area (cleanest signal); reuse order for the others.
ref_label = next(iter(area_data))
sarima_grid = sarima_aic_search(
    area_data[ref_label]['train'],
    p_range=(0, 1, 2), d_range=(0, 1), q_range=(0, 1, 2),
    P_range=(0, 1), D_range=(0, 1), Q_range=(0, 1),
    s=144, max_combos=18, subsample=144*7,
)
print(sarima_grid.head(10).to_string())
best_sarima_order    = sarima_grid.iloc[0]['order']
best_sarima_seasonal = sarima_grid.iloc[0]['seasonal_order']
print(f'\nSelected: order={best_sarima_order}, seasonal_order={best_sarima_seasonal}')

In [ ]:
from src.models.tuning import (
    random_search_neural, LSTM_SEARCH_SPACE, TRANSFORMER_SEARCH_SPACE,
)
from src.models.lstm_model import train_lstm, DEFAULT_CONFIG as LSTM_BASE
from src.models.transformer_model import train_transformer, DEFAULT_CONFIG as TRANS_BASE

# Use a smaller epoch budget for search; final training uses full epochs.
lstm_search_cfg  = {**LSTM_BASE,  'epochs': 8}
trans_search_cfg = {**TRANS_BASE, 'epochs': 8}

ref = area_data[ref_label]
print('LSTM random search:')
lstm_results_df = random_search_neural(
    train_lstm, ref['X_train'], ref['y_train'],
    base_config=lstm_search_cfg, search_space=LSTM_SEARCH_SPACE,
    n_trials=6, val_frac=0.1, seed=0,
)
best_lstm_cfg = {**LSTM_BASE, **lstm_results_df.iloc[0]['config'], 'epochs': 30}
print(f"\nLSTM chosen: {lstm_results_df.iloc[0]['config']}")

print('\nTransformer random search:')
trans_results_df = random_search_neural(
    train_transformer, ref['X_train'], ref['y_train'],
    base_config=trans_search_cfg, search_space=TRANSFORMER_SEARCH_SPACE,
    n_trials=6, val_frac=0.1, seed=0,
)
best_trans_cfg = {**TRANS_BASE, **trans_results_df.iloc[0]['config'], 'epochs': 30}
print(f"\nTransformer chosen: {trans_results_df.iloc[0]['config']}")

## 3.3 SARIMA Training

SARIMA(1,1,1)(1,1,1,144) — daily seasonality (s=144 = 1 day × 144 intervals of 10min).

> Note: SARIMA is computationally expensive at s=144. Consider reducing to s=48 (8 hours) if runtime is too high, or using a 1-week sample for hyperparameter selection.

In [ ]:
sarima_results = {}

for label, data in area_data.items():
    print(f'\n{"="*50}')
    print(f'SARIMA — {label}')
    result = run_sarima(
        data['train'], data['test'],
        order=best_sarima_order, seasonal_order=best_sarima_seasonal,
    )
    sarima_results[label] = result
    print(f"Timer: train={result['timer'].train_time:.1f}s | "
          f"inference={result['timer'].inference_time:.4f}s")

## 3.4 LSTM Training

In [ ]:
lstm_results = {}
for label, data in area_data.items():
    print(f'\n{"="*50}')
    print(f'LSTM — {label}')
    result = run_lstm(
        data['X_train'], data['y_train'],
        data['X_test'], data['y_test_orig'],
        data['scaler'], best_lstm_cfg,
    )
    lstm_results[label] = result
    torch.save(result, MODELS_DIR+f'lstm_{label}.pt')

## 3.5 Transformer Training

In [ ]:
transformer_results = {}
for label, data in area_data.items():
    print(f'\n{"="*50}')
    print(f'Transformer — {label}')
    result = run_transformer(
        data['X_train'], data['y_train'],
        data['X_test'], data['y_test_orig'],
        data['scaler'], best_trans_cfg,
    )
    transformer_results[label] = result
    torch.save(result, MODELS_DIR+f'transformer_{label}.pt')

## 3.6 Save All Results

In [ ]:
import pickle

all_results = {
    'persistence':    persistence_results,
    'seasonal_naive': seasonal_naive_results,
    'sarima':         sarima_results,
    'lstm':           lstm_results,
    'transformer':    transformer_results,
    'area_data':      {k: {'test': v['test'], 'y_test_orig': v['y_test_orig']}
                       for k, v in area_data.items()},
    'tuning': {
        'sarima_grid':        sarima_grid,
        'lstm_search':        lstm_results_df,
        'transformer_search': trans_results_df,
        'best_sarima':        {'order': best_sarima_order,
                               'seasonal_order': best_sarima_seasonal},
        'best_lstm':          best_lstm_cfg,
        'best_transformer':   best_trans_cfg,
    },
}

with open(MODELS_DIR+'all_results.pkl', 'wb') as f:
    pickle.dump(all_results, f)
print('All results saved.')